# 1. Data tables

Every table the monitoring layer reads, as a DataFrame. 

**Derived** (`backend/dashboard/models.py`): `MetricsDaily`, `MetricsParticipant`,
`MetricsCohort`, `Alert` - what `/api/monitor/*` serves. They hold no collected
data and are rebuilt by `manage.py recompute_metrics`.

**Raw** (`backend/app/models.py`): the ten `app_*` tables the `backend/dashboard/data/*`
modules actually import. `StressSample` and `EventDay` are excluded - nothing
under `dashboard/` reads either.

Read-only throughout. With `SYNTHETIC_DATA = True` no database is contacted.

In [1]:
# Set SYNTHETIC_DATA in monitor_common.py to swap fixture for ORM.
from monitor_common import *

print("data source:", describe_source())

data source: Django ORM - c5dqhursbgn9fb.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com / dd28mps21fsprq


In [2]:
# Every protocol constant comes from backend/dashboard/data/config.py, the single
# authority. Nothing in these notebooks redefines one as a literal.
pd.DataFrame(
    [{"constant": "STUDY_DAYS", "value": STUDY_DAYS},
     {"constant": "RUN_IN_DAYS", "value": RUN_IN_DAYS},
     {"constant": "PARTICIPANT_TZ", "value": str(PARTICIPANT_TZ)},
     {"constant": "WAKING_WINDOW (hours)", "value": f"{WAKING_WINDOW_START_HOUR}-{WAKING_WINDOW_END_HOUR}"},
     {"constant": "JITAI_COOLDOWN_MINUTES", "value": JITAI_COOLDOWN_MINUTES},
     {"constant": "DAILY_PROMPT_CAP", "value": DAILY_PROMPT_CAP},
     {"constant": "THRESHOLD_QUANTILE", "value": THRESHOLD_QUANTILE},
     {"constant": "MSSD_WINDOW", "value": MSSD_WINDOW},
     {"constant": "OUTCOME_WINDOW_HOURS", "value": OUTCOME_WINDOW_HOURS},
     {"constant": "RATE_MIN_PARTICIPANTS", "value": RATE_MIN_PARTICIPANTS},
     {"constant": "RATE_MIN_UNITS", "value": RATE_MIN_UNITS},
     {"constant": "BENCHMARKS", "value": str(BENCHMARKS)}])

,constant,value
0,STUDY_DAYS,35
1,RUN_IN_DAYS,7
2,PARTICIPANT_TZ,America/New_York
3,WAKING_WINDOW (hours),8-22
4,JITAI_COOLDOWN_MINUTES,60
5,DAILY_PROMPT_CAP,4
6,THRESHOLD_QUANTILE,0.8
7,MSSD_WINDOW,3
8,OUTCOME_WINDOW_HOURS,2
9,RATE_MIN_PARTICIPANTS,10


## Derived metric tables (`dashboard_*`)

In [16]:
# One row per participant per study day - the densest table, and what the
# Stage 2 heatmap is built from. A null here is STRUCTURAL, not zero: a day
# outside the participant's window carries is_active_day=False with every
# metric NULL, while an active day with no activity carries 0. Never fillna(0).
display(metrics_daily_df)

,id,user_id,study_day,local_date,is_run_in,is_active_day,computed_at,item_bank_version,ema_scheduled_n,ema_jitai_n,...,prompt_dismissed_n,outcome_captured_n,wear_valid_pct,wear_gap_pct,gaps_gt2h_n,max_gap_min,hr_minutes_valid,last_sync_age_h_eod,clock_skew_p95_ms,delivery_failures_n
0,3,430,0,2026-08-21,True,True,2026-09-16 01:40:14.416054+00:00,v1,1,0,...,0,0,None,None,None,None,None,None,122.0,0
1,4,430,1,2026-08-22,True,True,2026-09-16 01:40:14.682013+00:00,v1,0,0,...,0,0,None,None,None,None,None,None,NaN,0
2,5,430,2,2026-08-23,True,True,2026-09-16 01:40:14.947898+00:00,v1,0,0,...,0,0,None,None,None,None,None,None,NaN,0
3,6,430,3,2026-08-24,True,True,2026-09-16 01:40:15.216411+00:00,v1,0,0,...,0,0,None,None,None,None,None,None,NaN,0
4,7,430,4,2026-08-25,True,True,2026-09-16 01:40:15.489802+00:00,v1,0,0,...,0,0,None,None,None,None,None,None,NaN,0
5,8,430,5,2026-08-26,True,True,2026-09-16 01:40:15.758617+00:00,v1,0,0,...,0,0,None,None,None,None,None,None,55.0,0
6,9,430,6,2026-08-27,True,True,2026-09-16 01:40:16.139801+00:00,v1,2,0,...,0,0,None,None,None,None,None,None,636.0,0
7,10,430,7,2026-08-28,False,True,2026-09-16 01:40:16.411036+00:00,v1,0,0,...,0,0,None,None,None,None,None,None,NaN,0
8,11,430,8,2026-08-29,False,True,2026-09-16 01:40:16.682221+00:00,v1,0,0,...,0,0,None,None,None,None,None,None,NaN,0
9,12,430,9,2026-08-30,False,True,2026-09-16 01:40:16.953790+00:00,v1,0,0,...,0,0,None,None,None,None,None,None,NaN,0


In [4]:
# One row per participant. This is what orders the Stage 2 call list.
# risk_score is null, not zero, for anyone the study is not currently asking
# anything of (pre-enrolment, complete, withdrawn) - otherwise every term reads
# as maximally bad for them and they dominate the top of the list.
metrics_participant_df.head()

,id,user_id,computed_at,enrolled_at,day1_date,study_day_now,phase,is_enrolled_snapshot,first_seen_not_enrolled_at,last_ema_at,...,risk_components,slot_coverage_rate,slot_coverage_num,slot_coverage_den,prompt_response_rate,prompt_response_num,prompt_response_den,wear_rate,wear_num,wear_den
0,2,430,2026-09-18 05:39:44.105095+00:00,2026-08-21 04:57:38.306988+00:00,2026-08-21,28,mrt,True,None,2026-09-11 08:38:11.083288+00:00,...,"{'low_wear': 0, 'ema_stale': 15, 'sync_stale':...",0.011494,2,174,0.285714,2,7,None,0,0
1,1,531,2026-09-18 05:39:44.160157+00:00,2026-09-14 18:23:43.400385+00:00,2026-09-14,4,run_in,True,None,2026-09-17 17:33:52.561894+00:00,...,"{'low_wear': 0, 'ema_stale': 3, 'sync_stale': ...",0.300000,9,30,0.000000,0,2,None,0,0


In [5]:
# Append-only, one row per compute run per phase filter. cohort_latest is the
# row GET /api/monitor/cohort?phase= returns. benchmarks / series_14d /
# integrity / funnel stay JSON documents because the row is always read whole.
cohort_latest[["as_of", "phase_filter", "n_participants", "n_active"]]

,as_of,phase_filter,n_participants,n_active
0,2026-09-16 01:40:12.979822+00:00,phase2,2,2
1,2026-09-16 01:40:12.979822+00:00,phase1,0,0
2,2026-09-16 01:40:12.979822+00:00,all,2,2


In [6]:
# Open and resolved alerts. A null user_id is a COHORT-scoped alert, not
# missing data: a fact about the engine or the pipeline fires once for the
# cohort rather than once per participant, which would bury the view.
alerts_df[["fired_at", "severity", "rule_id", "user_id", "resolved_at"]]

,fired_at,severity,rule_id,user_id,resolved_at
0,2026-09-16 01:40:12.979822+00:00,critical,runin_violation,NaN,None
1,2026-09-16 01:40:12.979822+00:00,critical,cooldown_violation,430.0,None
2,2026-09-16 01:40:12.979822+00:00,warning,slot_coverage_low,430.0,None
3,2026-09-16 01:40:12.979822+00:00,high,no_ema_48h,430.0,None
4,2026-09-15 04:48:26.636207+00:00,critical,no_wearable_data,NaN,None
5,2026-09-15 04:48:26.636207+00:00,critical,randomization_audit,NaN,None
6,2026-09-15 04:48:26.636207+00:00,critical,sync_stale,NaN,None


## Raw source tables (`app_*`)

In [7]:
# Participants. email, names and password are deliberately not loaded - they
# are direct identifiers and no monitoring module reads them. The push token is
# reduced to a boolean because its presence is the only part the pipeline cares
# about: a missing token is why a prompt silently fails to deliver.
users_df

,user_id,is_enrolled,enrolled_at,gender,birthdate,has_push_token
0,430,True,2026-08-21 04:57:38.306988+00:00,other,2000-01-01,True
1,463,False,NaT,other,2000-01-01,True
2,464,False,NaT,other,2000-01-01,False
3,496,False,NaT,other,2000-01-01,True
4,497,False,NaT,other,2000-01-01,False
5,529,False,NaT,other,2000-01-01,True
6,530,False,NaT,other,2000-01-01,True
7,531,True,2026-09-14 18:23:43.400385+00:00,other,2000-01-01,True
8,532,False,NaT,other,2000-01-01,True


In [8]:
# One device per participant. WearableSync is the append-only log of the sync
# clock advancing; it exists because WearableDevice.last_synced_at is a single
# mutable column, so without it a sync outage cannot be told apart from genuine
# non-wear. In production nothing writes it yet - the fixture does.
print("devices:", len(wearable_devices_df), "| sync events:", len(wearable_sync_df))
wearable_devices_df.head()

devices: 2 | sync events: 0


,id,user_id,labfront_participant_id,is_active,last_synced_at
0,199,430,test-labfront-e2e-001,True,None
1,232,531,TEST-531,True,None


In [9]:
# Every check-in. served_sub_item_ids records what actually reached the screen,
# which is what makes item completeness measurable at all. No free-text field is
# stored anywhere in this table - that is an IRB constraint, not an omission.
print(ema_df.groupby(["ema_type", "status"]).size().to_string())
ema_df.head()

ema_type            status   
post_prompt         completed     2
prompt_feedback     completed     6
scheduled_check_in  completed    24


,id,user_id,prompt_id,ema_type,status,sent_at,responded_at,expires_at,outcome_window_start,outcome_window_end,source_jitai_log_id,served_sub_item_ids,mood,stress,energy
0,8494,430,EMA-430-20260821045625,scheduled_check_in,completed,2026-08-21 04:57:38.306988+00:00,2026-08-21 04:57:38.306686+00:00,NaT,NaT,NaT,NaN,None,None,None,None
1,8521,430,EMA-430-20260827210253,scheduled_check_in,completed,2026-08-27 21:03:15.495740+00:00,2026-08-27 21:03:15.495497+00:00,NaT,NaT,NaT,NaN,None,None,None,None
2,8522,430,EMA-430-20260827231630,scheduled_check_in,completed,2026-08-27 23:16:52.774014+00:00,2026-08-27 23:16:52.773372+00:00,NaT,NaT,NaT,NaN,None,None,None,None
3,8548,430,EMA-JITAI-199,post_prompt,completed,2026-09-06 22:24:19.525166+00:00,2026-09-06 22:24:19.524855+00:00,2026-09-07 00:15:53.529848+00:00,2026-09-06 22:15:53.529848+00:00,2026-09-07 00:15:53.529848+00:00,199.0,None,None,None,None
4,8554,430,EMA-C0-199,prompt_feedback,completed,2026-09-11 08:13:56.769608+00:00,2026-09-11 08:13:56.769324+00:00,NaT,NaT,NaT,199.0,None,None,None,None


In [10]:
# One row per answered sub-item. Joined to the frozen item bank (v1), this is
# what Stage 3's completeness matrix scores: answered over askable.
ema_item_responses_df.head()

,id,ema_id,item_id,sub_item_id,response_type,value_numeric,value_choice,value_choices
0,56,8494,B1,B1_affect_angry,likert,5.0,None,None
1,55,8494,B1,B1_affect_anxious,likert,5.0,None,None
2,60,8494,B1,B1_affect_bored,likert,5.0,None,None
3,58,8494,B1,B1_affect_excited,likert,5.0,None,None
4,59,8494,B1,B1_affect_happy,likert,5.0,None,None


In [11]:
# The reminder log. It is what lets a missed slot be classified: covered,
# reminded-but-skipped, or silent. Only the third is a scheduler failure, and
# nobody should be phoned about it.
checkin_reminders_df.head()

,id,user_id,sent_at,daily_count_at_send
0,1,430,2026-08-27 13:01:58.870266+00:00,0
1,2,430,2026-08-27 15:04:58.865991+00:00,0
2,3,430,2026-08-27 17:07:58.860053+00:00,0
3,4,430,2026-08-27 19:10:58.835019+00:00,0
4,5,430,2026-08-27 21:13:58.848392+00:00,1


In [12]:
# Every decision point, sent or not - send_prompt is the gate, so the full
# table is the denominator and the send_prompt=True subset is the dose.
# threshold_at_decision records what observed_mssd was compared against;
# threshold_source is 'engine' when the live decision wrote it.
print("decision points:", len(jitai_log_df), "| sent:", int(jitai_log_df["send_prompt"].sum()))
print(jitai_log_df["threshold_source"].value_counts(dropna=False).to_string())
jitai_log_df.head()

decision points: 39 | sent: 27
threshold_source
          33
engine     6


,id,user_id,prompt_id,triggered_at,trigger_reason,trigger_signal,decision_point_id,ema_id,observed_mssd,threshold_at_decision,...,receipt_event_id,delivery_status,delivery_error,receipt_platform,receipt_app_state,eligible_prompt_ids,evaluated_items,matched_categories,category_drawn,fallback_reason
0,139,430,PROMPT-SIM-1787288405457,2026-08-21 05:00:05.652025+00:00,hr_elevated+stress_high,None,None,NaN,NaN,NaN,...,None,received_on_device,,android,foreground,None,None,None,None,
1,167,430,default,2026-08-27 04:07:19.469257+00:00,manual test push,None,None,NaN,NaN,NaN,...,None,received_on_device,,android,foreground,None,None,None,None,
2,168,430,default,2026-08-27 04:07:53.444227+00:00,manual test push,None,None,NaN,NaN,NaN,...,None,accepted_by_expo,,,,None,None,None,None,
3,169,430,,2026-08-27 04:07:58.725779+00:00,missing or insufficient EMA data,None,ema_8494,8494.0,NaN,NaN,...,None,not_sent,,,,None,None,None,None,
4,170,430,default,2026-08-27 04:09:29.704116+00:00,manual test push,None,None,NaN,NaN,NaN,...,None,accepted_by_expo,,,,None,None,None,None,


In [13]:
# Engagement backs the opened / acted / dismissed counts. PhoneTelemetry
# carries occurred_at (device clock) beside recorded_at (server clock); their
# difference is clock skew, which is SIGNED - a device running ahead of the
# server is a real diagnostic condition, not an error.
print(engagement_log_df["event_type"].value_counts().to_string())
phone_telemetry_df.head()

event_type
ema_opened             76
ema_dismissed          40
ema_completed          32
notification_tapped    23


,id,user_id,session_id,event_type,occurred_at,recorded_at,screen_name,latency_ms,metadata
0,274,430,08b8eb82-4d85-4364-b34a-9b72fe39fb8f,draft_started,2026-08-21 05:00:07.083000+00:00,2026-08-21 05:00:07.205154+00:00,compose,None,"{'delete_count': 0, 'keystroke_count': 1, 'tim..."
1,311,430,fa137c4e-f6d2-4b44-b205-b340dc893e76,draft_started,2026-08-27 03:37:14.330000+00:00,2026-08-27 03:37:14.379701+00:00,compose,None,"{'delete_count': 0, 'keystroke_count': 1, 'tim..."
2,312,430,fa137c4e-f6d2-4b44-b205-b340dc893e76,draft_deleted,2026-08-27 03:37:43.076000+00:00,2026-08-27 03:37:43.131435+00:00,compose,None,"{'delete_count': 0, 'keystroke_count': 1, 'tim..."
3,313,430,ead76c5c-e58d-40fc-be3d-1cc80b0a40fc,draft_started,2026-08-27 03:37:45.474000+00:00,2026-08-27 03:37:45.519285+00:00,compose,None,"{'delete_count': 0, 'keystroke_count': 1, 'tim..."
4,314,430,ead76c5c-e58d-40fc-be3d-1cc80b0a40fc,draft_deleted,2026-08-27 03:37:49.536000+00:00,2026-08-27 03:37:49.578201+00:00,compose,None,"{'delete_count': 0, 'keystroke_count': 2, 'tim..."


In [14]:
# The only table that can run to millions of rows - production ingests heart
# rate every 15 seconds. HR_DAYS bounds it to a trailing window; the full count
# prints beside the loaded count so the truncation is always visible.
print(f"rows in table: {hr_total} | loaded (HR_DAYS={HR_DAYS}): {len(heart_rate_df)}")
heart_rate_df.head()

rows in table: 0 | loaded (HR_DAYS=14): 0


,id,user_id,timestamp,bpm,source


## Summary

In [15]:
# Every frame loaded, with its shape. An all-zero derived block means
# recompute_metrics has never run against this database.
table_summary()

,table,rows,columns
0,dashboard_metricsdaily,34,39
1,dashboard_metricsparticipant,2,23
2,dashboard_metricscohort,375,17
3,dashboard_alert,7,7
4,app_user,9,6
5,app_wearabledevice,2,5
6,app_wearablesync,0,6
7,app_ema,32,15
8,app_emaitemresponse,722,8
9,app_checkinreminder,155,4
